# sources

> Grow the concept graph: from any text, from a vault document, and from vedicreader.

This module adds concepts to the `learning` shelf. A model finds the core ideas in a text, and code merges them with the concepts on the shelf.
The vedicreader import reads the repo files and needs no model.
The tutor then asks about the concepts that this module adds.

In [ ]:
#| default_exp sources

In [ ]:
#| hide
from fastcore.test import *

In [ ]:
#| hide
import os, tempfile
os.environ['VISHALAKSHI_VAULT'] = os.environ['ABHYASA_VAULT'] = os.path.join(tempfile.mkdtemp(), 'vault.db')   # a test that opens the default vault gets a temporary one

In [ ]:
#| export
import json, os, re
from pathlib import Path
from fastcore.utils import *
from abhyasa.store import Learning, slugify
from abhyasa.llm import ask_json

## Extraction from text

The model lists the core ideas of a text. Each idea has a true, self-contained statement and its prereqs.
Code does the merge. When a slug or an alias matches a concept on the shelf, it is the same concept.
Its lists grow and its learner state stays.

`EXTRACT_SP` and `extract_prompt` ask the model for the concepts.

In [ ]:
#| export
EXTRACT_SP = """You find the core ideas in a text, for a learner who will be tested on them.
- A concept is one idea a learner can be asked about: a definition, a rule, a method, a result, a distinction.
- `statement` is true and self-contained: it makes sense without the text. One to three sentences. Maths in $...$.
- Skip trivia, examples and names that are not ideas. Prefer 4-15 concepts for a page or two.
- `prereqs` names other concepts (from your list or from the known list) that a learner must know first."""

def extract_prompt(text, subject='', known=()):
    kn = ', '.join(known[:150])
    return ((f"Subject: {subject}\n" if subject else '') + (f"Concepts the learner already has: {kn}\n" if kn else '')
            + f"\nText:\n{text}\n\n"
            'Return JSON: {"concepts": [{"name": str, "statement": str, "aliases": [str], "prereqs": [str]}]}')

`_index` finds the slug for a name or an alias. `_windows` cuts a long text into parts for the model.

In [ ]:
#| export
def _index(lr):
    "Map each lower-case name, alias and slug on the shelf to its slug."
    out = {}
    for c in lr.concepts():
        for n in (c.slug, c.name, *(c.aliases or ())): out[n.lower()] = c.slug
    return out

def _windows(text, size=12000):
    "Cut a long text at paragraph breaks into windows of about `size` characters."
    out, cur = [], ''
    for p in re.split(r'\n\s*\n', text):
        if cur and len(cur) + len(p) > size: out.append(cur); cur = ''
        cur += p + '\n\n'
    return [*out, cur] if cur.strip() else out

`extract` adds the concepts of a text. `from_vault` does the same for one document of the vault.

In [ ]:
#| export
def extract(lr:Learning, ask, text:str, subject:str='', source:dict=None) -> L:
    "Add the core ideas of `text` to the shelf as concepts, merged with the concepts there. Return the slugs that it changed."
    touched = L()
    for w in _windows(text):
        idx = _index(lr)
        r = ask_json(ask, extract_prompt(w, subject, sorted({v for v in idx.values()})), EXTRACT_SP)
        cs = r.get('concepts', []) if isinstance(r, dict) else r
        slug_of = {}
        for c in cs:
            names = [c['name'], *(c.get('aliases') or ())]
            slug_of[c['name'].lower()] = next((idx[n.lower()] for n in names if n.lower() in idx), slugify(c['name']))
        for c in cs:
            s = slug_of[c['name'].lower()]
            pre = [slug_of.get(p.lower()) or idx.get(p.lower()) for p in c.get('prereqs') or ()]
            lr.put(c['name'], c.get('statement', ''), subject, slug=s, aliases=c.get('aliases') or (),
                   prereqs=[p for p in pre if p], provenance=[source] if source else ())
            touched.append(s)
    return touched.unique()

def from_vault(lr:Learning, ask, ref:str, subject:str='') -> L:
    "Extract concepts from one document of the main shelf of the vault, by doc id, source or title."
    d = lr.vault.document(ref)
    return extract(lr, ask, d.text, subject, dict(src='vault', doc_id=d.doc_id, title=d.title))

A fake model returns three concepts. One matches `Ranking` on the shelf by its alias `rank`.

In [ ]:
from abhyasa.store import temp_learning
from abhyasa.llm import Fake
lr = temp_learning()
lr.put('Ranking', 'Order items by a score.', 'ml', aliases=['rank'])
fake = Fake({'Return JSON: {"concepts"': dict(concepts=[
    dict(name='Reciprocal rank fusion', statement='Sum 1/(k + rank) over lists.', aliases=['RRF'], prereqs=['rank', 'Ranked list']),
    dict(name='Ranked list', statement='Items in score order.', aliases=[], prereqs=[]),
    dict(name='Ranking', statement='Ordering by score.', aliases=['ordering'], prereqs=[])])})

`extract` returns the changed slugs. The prereqs become slugs, and the old concept gets the new alias and statement.

In [ ]:
test_eq(sorted(extract(lr, fake, 'some text', 'ml', dict(src='test'))), ['ranked-list', 'ranking', 'reciprocal-rank-fusion'])
test_eq(lr.concept('reciprocal-rank-fusion').prereqs, ['ranking', 'ranked-list'])
test_eq(lr.concept('ranking').aliases, ['rank', 'ordering'])
test_eq(lr.concept('ranking').statement, 'Ordering by score.')

`_windows` cuts at paragraph breaks.

In [ ]:
test_eq(len(_windows('a\n\n' * 10, size=5)), 5)

## vedicreader

Read the files of the vedicreader repo. Do not read the website. `$VEDICREADER_ROOT` gives the path of the repo.

| source | concepts |
|---|---|
| Sanskrit lessons | One concept for each lesson. The lesson before it in the course is its prereq. Its `check` pairs become ready questions at depth 4 (apply). Its recap becomes the statement. |
| `static/grammar/terms/*.json` | One concept for each term, with `related` links. A term with a jyotiṣa term of the same name also links to it. |
| `vr/jyotisha/explain/*.json` | One concept for each graha, rāśi and nakṣatra. |

`VR_ROOT` is the repo path. `plain` removes the marks of vedicreader from a text.

In [ ]:
#| export
VR_ROOT = Path(os.getenv('VEDICREADER_ROOT', Path.home()/'code/personal/orgs/vedicreader'))

def plain(s:str) -> str:
    "The vedicreader text without its `{…}` marks and `[1,2]` citations."
    return re.sub(r'\[\d+(?:,\s*\d+)*\]', '', re.sub(r'\{([^{}]*)\}', r'\1', s or '')).strip()

Each of these three generators reads one source and yields the concepts as dicts for `put`.

In [ ]:
#| export
def _lesson_concepts(root):
    d = root/'static/learn'
    order, prev = json.loads((d/'order.json').read_text()), None
    for ui, u in enumerate(order['units']):
        for li, ls in enumerate(u['lessons']):
            p = d/'lessons'/f"{ls['slug']}.json"
            body = json.loads(p.read_text()).get('body', []) if p.exists() else []
            recap = [plain(x) for b in body for x in b.get('recap', [])]
            qs = [dict(q=plain(c['q']), a=plain(c['a']), depth=4) for b in body for c in b.get('check', [])]
            yield dict(name=ls['title'], statement=' '.join([plain(ls['cap']), *recap]), subject='sanskrit', slug=f"sa-{ls['slug']}",
                       prereqs=[prev] if prev else [], questions=qs, order=ui*100 + li, unit=u['name'],
                       provenance=[dict(src='vedicreader', lesson=ls['slug'], url=f"https://vedicreader.com/v/learn/{ls['slug']}")])
            prev = f"sa-{ls['slug']}"

In [ ]:
#| export
def _term_concepts(root):
    for p in sorted((root/'static/grammar/terms').glob('*.json')):
        for t in json.loads(p.read_text()).get('terms', []):
            en = ', '.join(t.get('en') or [])
            yield dict(name=t.get('iast') or t['slug'], statement=plain(t.get('defn', '')), subject='sanskrit', slug=f"sa-term-{t['slug']}",
                       aliases=[a for a in (t.get('sa'), *(t.get('en') or ())) if a], group=t.get('group', ''), order=1000,
                       related=[f"sa-term-{r}" for r in t.get('related') or ()] + ([f"jy-term-{t['jy']}"] if t.get('jy') else []),
                       provenance=[dict(src='vedicreader', term=t['slug'], cite=(t.get('src') or {}).get('cite', ''))],
                       questions=[dict(q=f"What is {t.get('iast') or t['slug']} ({en})? Say what it is and where it is used.",
                                       a=plain(t.get('defn', '')), depth=2)] if t.get('defn') else [])

In [ ]:
#| export
RASI = [('Meṣa', 'Aries'), ('Vṛṣabha', 'Taurus'), ('Mithuna', 'Gemini'), ('Karkaṭa', 'Cancer'), ('Siṃha', 'Leo'), ('Kanyā', 'Virgo'),
        ('Tulā', 'Libra'), ('Vṛścika', 'Scorpio'), ('Dhanus', 'Sagittarius'), ('Makara', 'Capricorn'), ('Kumbha', 'Aquarius'), ('Mīna', 'Pisces')]

def _explain_name(kind, key):
    "The name and aliases of one explained item; the rāśi file keys each sign by its number, 0 for Meṣa."
    if kind == 'rasi' and key.isdigit(): nm, en = RASI[int(key)]; return nm, [en, 'rāśi', 'sign']
    return key, [kind]

def _explain_concepts(root):
    for kind in ('graha', 'rasi', 'nakshatra'):
        p = root/f'vr/jyotisha/explain/{kind}.json'
        if not p.exists(): continue
        for name, v in json.loads(p.read_text()).items():
            secs = v.get('sections', {})
            first_sec = next(iter(secs.values()), [])
            st = plain(first_sec[0] if first_sec else '')[:900]
            nm, al = _explain_name(kind, name)
            yield dict(name=nm, statement=st, subject='jyotisha', slug=f"jy-{kind}-{slugify(nm)}", kind=kind, order=2000,
                       aliases=al, provenance=[dict(src='vedicreader', explain=kind, key=name)])

`from_vedicreader` runs the generators that `parts` names and returns a count for each part.

In [ ]:
#| export
def from_vedicreader(lr:Learning, root=None, parts=('lessons', 'terms', 'explain')) -> dict:
    "Add the lessons, terms and jyotiṣa explanations of vedicreader as concepts. Keep the learner state on the shelf."
    root, n = Path(root or VR_ROOT), {}
    gens = dict(lessons=_lesson_concepts, terms=_term_concepts, explain=_explain_concepts)
    for part in parts:
        n[part] = 0
        for c in gens[part](root):
            lr.put(**c); n[part] += 1
    return n

`plain` removes the marks and the citations.

In [ ]:
test_eq(plain('{a} + {a} → {ā}: {nāsti}.[1,2]'), 'a + a → ā: nāsti.')

When the repo is on this machine, the lessons become a chain of concepts with prereqs and ready questions.

In [ ]:
if VR_ROOT.exists():
    lr = temp_learning()
    n = from_vedicreader(lr, parts=('lessons',))
    assert n['lessons'] > 10
    c = lr.concept('sa-sandhi-vowels')
    assert c.prereqs and c.questions and c.subject == 'sanskrit'
    test_eq(lr.concept('sa-sounds').prereqs, [])
    test_eq(_explain_name('rasi', '0'), ('Meṣa', ['Aries', 'rāśi', 'sign']))

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()